In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 16)
spark.conf.set("spark.sql.files.maxPartitionBytes", 134217728)
spark.conf.set("spark.sql.files.openCostInBytes", 134217728)

In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.utils import AnalysisException
import logging

logging.basicConfig(level=logging.INFO)

logger = logging.getLogger("BronzePipeline")

In [0]:
CATALOG = "main_workspace"
BRONZE_SCHEMA = "bronze"

spark.sql(f"USE CATALOG {CATALOG}")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

DataFrame[]

In [0]:
BASE_PATH = "s3://e-commerce-analysis-pipeline/processed"

DATASETS = {
    "customers": f"{BASE_PATH}/customer_parquet/",
    "order_items": f"{BASE_PATH}/order_items_parquet/",
    "payments": f"{BASE_PATH}/order_payment_dataset_parquet/",
    "orders": f"{BASE_PATH}/orders_datasets_parquet/",
    "product_category_translation": f"{BASE_PATH}/product_category_name_translation_parquet/",
    "products": f"{BASE_PATH}/product_dataset_parquet/",
    "sellers": f"{BASE_PATH}/sellers_datasets/"
}

In [0]:
def load_to_bronze(table_name, path):

    logger.info(f"Starting ingestion for {table_name}")

    try:

        df = spark.read.format("parquet").load(path)

        # Check if dataframe is empty (Spark Connect compatible)
        if df.limit(1).count() == 0:
            logger.warning(f"{table_name} dataset is empty")
            return

        df = df.withColumn("ingestion_timestamp", current_timestamp())

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
        )

        logger.info(f"{table_name} successfully loaded")

    except AnalysisException as ae:

        logger.error(f"Schema error while loading {table_name}")
        logger.error(str(ae))

    except Exception as e:

        logger.error(f"Unexpected error while loading {table_name}")
        logger.error(str(e))

In [0]:
for table, path in DATASETS.items():
    load_to_bronze(table, path)

INFO:BronzePipeline:Starting ingestion for customers
INFO:BronzePipeline:customers successfully loaded
INFO:BronzePipeline:Starting ingestion for order_items
INFO:BronzePipeline:order_items successfully loaded
INFO:BronzePipeline:Starting ingestion for payments
INFO:BronzePipeline:payments successfully loaded
INFO:BronzePipeline:Starting ingestion for orders
INFO:BronzePipeline:orders successfully loaded
INFO:BronzePipeline:Starting ingestion for product_category_translation
INFO:BronzePipeline:product_category_translation successfully loaded
INFO:BronzePipeline:Starting ingestion for products
INFO:BronzePipeline:products successfully loaded
INFO:BronzePipeline:Starting ingestion for sellers
INFO:BronzePipeline:sellers successfully loaded


In [0]:
spark.sql("SHOW TABLES IN main_workspace.bronze").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|  bronze|           customers|      false|
|  bronze|         order_items|      false|
|  bronze|              orders|      false|
|  bronze|            payments|      false|
|  bronze|product_category_...|      false|
|  bronze|            products|      false|
|  bronze|             sellers|      false|
+--------+--------------------+-----------+



In [0]:
spark.sql("""
SELECT *
FROM main_workspace.bronze.orders
LIMIT 10
""").display()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2026-03-12T04:53:31.464015Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2026-03-12T04:53:31.464015Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2026-03-12T04:53:31.464015Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2026-03-12T04:53:31.464015Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2026-03-12T04:53:31.464015Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,2026-03-12T04:53:31.464015Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,,,2017-05-09,2026-03-12T04:53:31.464015Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,2026-03-12T04:53:31.464015Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,2026-03-12T04:53:31.464015Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,2026-03-12T04:53:31.464015Z


In [0]:
spark.sql("""
DESCRIBE DETAIL main_workspace.bronze.orders
""").display()

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,67e26cc1-74e9-41a2-9e87-f86b0289c356,main_workspace.bronze.orders,null,s3://databricks-storage-7474651073079039/unity-catalog/7474651073079039/__unitystorage/catalogs/e3276075-8b08-4d53-8c19-3cc54df42bb9/tables/3a096a70-113a-439b-8bda-0c6c7ffa2745,2026-03-12T04:53:30.629Z,2026-03-12T04:53:34Z,List(),List(),1,6195139,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
spark.sql("""
DESCRIBE HISTORY main_workspace.bronze.orders
""").display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-03-12T04:53:34Z,71914497577424,indrasenareddy46372@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3963836697283453),0311-095412-qc8n38cv,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 6195139)",null,Databricks-Runtime/17.3.x-photon-scala2.13


# **--Silver Layer--**